# Job Dimension (SCD Type 2) Loader

This notebook maintains the `warehouse.dim_job` dimension table with SCD Type 2 logic using standardized batch processing.

**Purpose**: Track job posting changes over time with effective date windows

**Key Features**:
* **SCD Type 2** for tracking historical changes to job postings
* Stable surrogate keys (job_sk) for each version
* Enterprise job ID as business key
* Foreign key resolution to company, location, sector, role dimensions
* Effective date windows (effective_from, effective_to, is_current)
* Change detection via record hash comparison
* Automatic version management (close old, insert new)

**Architecture**:
- **Source**: Silver layer current jobs (`workspace.silver.silver_jobs_current`)
- **Reference**: Dimension tables (company, location, sector, role)
- **Target**: `workspace.warehouse.dim_job`
- **Metadata**: `workspace.metadata.dim_job_refresh_log`

**SCD2 Processing Logic**:
1. Extract active jobs from silver layer
2. Resolve foreign keys to all dimension tables
3. Detect changes by comparing record hashes
4. Close existing current records (set effective_to, is_current=FALSE)
5. Insert new records and new versions (generate new job_sk)
6. Track processing metrics in metadata log

**Idempotency**: Safe to re-run; uses change detection to avoid duplicates

In [0]:
dbutils.widgets.text("force_full_refresh", "false", "Force Full Refresh (true/false)")

force_full_refresh = dbutils.widgets.get("force_full_refresh").strip().lower() == "true"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DoubleType, DecimalType, TimestampType, BooleanType
from pyspark.sql.window import Window
from datetime import datetime
import json

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
INTERMEDIATE_SCHEMA = f"{CATALOG}.intermediate"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Source and target tables
SOURCE_JOBS_TABLE = f"{SILVER_SCHEMA}.silver_jobs_current"
COMPANY_ALIAS_TABLE = f"{WAREHOUSE_SCHEMA}.dim_company_alias"
COMPANY_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_company"
LOCATION_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_location"
SECTOR_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_sector"
ROLE_MAP_TABLE = f"{INTERMEDIATE_SCHEMA}.inter_job_role_map"
ROLE_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_role"
TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_job"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_job_refresh_log"

# Generate run ID
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = F.current_timestamp()

print(f"Run ID: {run_id}")
print(f"Source table: {SOURCE_JOBS_TABLE}")
print(f"Force full refresh: {force_full_refresh}")

In [0]:
# Define target table schema (SCD Type 2)
target_schema = StructType([
    StructField("job_sk", LongType(), False, metadata={"comment": "Surrogate key for job version"}),
    StructField("enterprise_job_id", StringType(), False, metadata={"comment": "Business key - canonical job ID"}),
    StructField("source_name", StringType(), True, metadata={"comment": "Source system name"}),
    StructField("source_job_id", StringType(), True, metadata={"comment": "Source system job ID"}),
    StructField("canonical_role_id", StringType(), True, metadata={"comment": "Canonical role identifier"}),
    StructField("company_sk", LongType(), False, metadata={"comment": "Foreign key to dim_company"}),
    StructField("location_sk", LongType(), False, metadata={"comment": "Foreign key to dim_location"}),
    StructField("sector_sk", LongType(), False, metadata={"comment": "Foreign key to dim_sector"}),
    StructField("role_sk", LongType(), False, metadata={"comment": "Foreign key to dim_role"}),
    StructField("title_normalized", StringType(), True, metadata={"comment": "Normalized job title"}),
    StructField("description_raw", StringType(), True, metadata={"comment": "Raw job description"}),
    StructField("remote_type", StringType(), True, metadata={"comment": "Remote work type"}),
    StructField("posted_at", TimestampType(), True, metadata={"comment": "Job posting date"}),
    StructField("last_seen", TimestampType(), True, metadata={"comment": "Last seen date"}),
    StructField("effective_from", TimestampType(), False, metadata={"comment": "SCD2 effective start date"}),
    StructField("effective_to", TimestampType(), True, metadata={"comment": "SCD2 effective end date (NULL = current)"}),
    StructField("is_current", BooleanType(), False, metadata={"comment": "Is this the current version"}),
    StructField("record_hash", StringType(), True, metadata={"comment": "Hash for change detection"})
])

# Create target table if not exists
if not spark.catalog.tableExists(TARGET_TABLE):
    print(f"Creating target table: {TARGET_TABLE}")
    spark.createDataFrame([], target_schema).write.format("delta").saveAsTable(TARGET_TABLE)
    print("✓ Target table created")
else:
    print(f"Target table exists: {TARGET_TABLE}")

# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("jobs_extracted", IntegerType(), True),
    StructField("new_jobs_inserted", IntegerType(), True),
    StructField("jobs_updated", IntegerType(), True),
    StructField("old_records_closed", IntegerType(), True),
    StructField("no_change_jobs", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

# Create metadata table if not exists
if not spark.catalog.tableExists(METADATA_TABLE):
    print(f"Creating metadata table: {METADATA_TABLE}")
    spark.createDataFrame([], metadata_schema).write.format("delta").saveAsTable(METADATA_TABLE)
    print("✓ Metadata table created")
else:
    print(f"Metadata table exists: {METADATA_TABLE}")

In [0]:
# Define sector classification function
def classify_sector_by_keywords(title, sector_keywords_dict):
    """
    Classify a job title into a sector by matching against sector keywords.
    
    Args:
        title: Job title (normalized)
        sector_keywords_dict: Dict mapping sector_sk to list of keywords
    
    Returns:
        sector_sk: Matched sector or -1 if no match
    """
    if not title:
        return -1
    
    title_lower = title.lower()
    
    # Score each sector by keyword matches
    sector_scores = {}
    for sector_sk, keywords in sector_keywords_dict.items():
        if sector_sk == -1:  # Skip "Unknown" sector
            continue
        
        score = 0
        for keyword in keywords:
            if keyword.lower() in title_lower:
                score += 1
        
        if score > 0:
            sector_scores[sector_sk] = score
    
    # Return sector with highest score
    if sector_scores:
        best_sector = max(sector_scores.items(), key=lambda x: x[1])[0]
        return int(best_sector)
    
    return -1

print("Sector classification function defined")

In [0]:
print("Extracting active jobs from silver layer...", end=" ")

# Load active jobs from silver layer
jobs_df = spark.table(SOURCE_JOBS_TABLE).filter(
    (F.col("enterprise_job_id").isNotNull()) &
    (F.col("is_active") == True) &
    (F.col("soft_delete_flag") == False)
)

jobs_count = jobs_df.count()
print(f"✓ Extracted {jobs_count} active jobs")

print("Resolving foreign keys...", end=" ")

# Load dimension tables for FK resolution
company_alias_df = spark.table(COMPANY_ALIAS_TABLE)
company_df = spark.table(COMPANY_DIM_TABLE)
location_df = spark.table(LOCATION_DIM_TABLE)
sector_df = spark.table(SECTOR_DIM_TABLE)
role_map_df = spark.table(ROLE_MAP_TABLE)
role_df = spark.table(ROLE_DIM_TABLE)

# Build sector keywords dictionary for classification
sector_keywords_rows = sector_df.select("sector_sk", "keywords").collect()
sector_keywords_dict = {row.sector_sk: row.keywords if row.keywords else [] for row in sector_keywords_rows}
print(f"Loaded {len(sector_keywords_dict)} sectors with keywords")

# Resolve company_sk via company alias
jobs_with_company = jobs_df.alias("j").join(
    company_alias_df.alias("ca"),
    F.col("j.company_name_norm") == F.col("ca.alias_name"),
    "left"
).join(
    company_df.alias("dc"),
    F.col("ca.company_sk") == F.col("dc.company_sk"),
    "left"
).select(
    F.col("j.*"),
    F.coalesce(F.col("dc.company_sk"), F.lit(-1)).alias("company_sk")
)

# Resolve location_sk
jobs_with_location = jobs_with_company.alias("j").join(
    location_df.alias("dl"),
    F.col("j.location_norm") == F.col("dl.location_name"),
    "left"
).select(
    F.col("j.*"),
    F.coalesce(F.col("dl.location_sk"), F.lit(-1)).alias("location_sk")
)

# Resolve sector_sk via keyword matching on job title

# Create UDF for sector classification
classify_sector_udf = F.udf(
    lambda title: classify_sector_by_keywords(title, sector_keywords_dict),
    LongType()
)

jobs_with_sector = jobs_with_location.withColumn(
    "sector_sk",
    classify_sector_udf(F.col("title_normalized"))
)

# Resolve role_sk via semantic role mapping
jobs_with_role = jobs_with_sector.alias("j").join(
    role_map_df.alias("rm"),
    F.col("j.title_normalized") == F.col("rm.title_normalized"),
    "left"
).join(
    role_df.alias("dr"),
    F.col("rm.canonical_role_id") == F.col("dr.canonical_role_id"),
    "left"
).select(
    F.col("j.*"),
    F.col("rm.canonical_role_id"),
    F.coalesce(F.col("dr.role_sk"), F.lit(-1)).alias("role_sk")
)

print("✓ Foreign keys resolved")

# Select final columns for processing
jobs_extract = jobs_with_role.select(
    "enterprise_job_id",
    "source_name",
    "source_job_id",
    "company_sk",
    "location_sk",
    "sector_sk",
    "role_sk",
    "canonical_role_id",
    "title_normalized",
    "description_raw",
    "remote_type",
    "posted_at",
    "last_seen",
    "record_hash",
    "updated_at"
)

In [0]:
print("Detecting changes for SCD2 processing...", end=" ")

# Load existing dimension with current records
existing_jobs = spark.table(TARGET_TABLE).filter(
    F.col("is_current") == True
).select(
    "job_sk",
    "enterprise_job_id",
    "record_hash",
    "is_current"
)

# Join to detect changes
jobs_with_changes = jobs_extract.alias("j").join(
    existing_jobs.alias("e"),
    F.col("j.enterprise_job_id") == F.col("e.enterprise_job_id"),
    "left"
).select(
    F.col("j.*"),
    F.col("e.job_sk").alias("existing_job_sk"),
    F.col("e.record_hash").alias("existing_hash"),
    F.col("e.is_current").alias("existing_is_current")
).withColumn(
    "change_type",
    F.when(
        F.col("existing_job_sk").isNull(),
        F.lit("INSERT")
    ).when(
        (F.col("record_hash") != F.col("existing_hash")) & (F.col("existing_is_current") == True),
        F.lit("UPDATE")
    ).otherwise(
        F.lit("NOCHANGE")
    )
)

# Count changes
change_summary = jobs_with_changes.groupBy("change_type").count().collect()
change_counts = {row["change_type"]: row["count"] for row in change_summary}

inserts = change_counts.get("INSERT", 0)
updates = change_counts.get("UPDATE", 0)
no_change = change_counts.get("NOCHANGE", 0)

print(f"✓ Changes detected: {inserts} new, {updates} updates, {no_change} no change")

In [0]:
print(f"Processing SCD2 updates into {TARGET_TABLE}... ", end="")

try:
    # Step 1: Close old records for updated jobs
    if updates > 0:
        records_to_close = jobs_with_changes.filter(
            F.col("change_type") == "UPDATE"
        ).select(
            F.col("existing_job_sk").alias("job_sk"),
            F.col("updated_at").alias("effective_to")
        ).distinct()
        
        records_to_close.createOrReplaceTempView("records_to_close")
        
        close_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING records_to_close source
        ON target.job_sk = source.job_sk
          AND target.is_current = TRUE
        WHEN MATCHED THEN UPDATE SET
          target.effective_to = source.effective_to,
          target.is_current = FALSE
        """
        
        spark.sql(close_sql)
        print(f"[Closed {updates} old records] ", end="")
    
    # Step 2: Prepare new records (inserts + new versions of updates)
    new_and_updated = jobs_with_changes.filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    
    if new_and_updated.count() > 0:
        # Get current max job_sk
        max_sk_result = spark.sql(f"SELECT COALESCE(MAX(job_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
        max_sk = max_sk_result[0]['max_sk']
        
        # Generate new surrogate keys
        window_spec = Window.orderBy("enterprise_job_id", "updated_at")
        
        records_to_insert = new_and_updated.withColumn(
            "job_sk",
            (F.row_number().over(window_spec) + max_sk).cast(LongType())
        ).withColumn(
            "effective_from",
            F.col("updated_at")
        ).withColumn(
            "effective_to",
            F.lit(None).cast(TimestampType())
        ).withColumn(
            "is_current",
            F.lit(True)
        ).withColumn(
            "salary_min",
            F.lit(None).cast(DecimalType(18, 2))
        ).withColumn(
            "salary_max",
            F.lit(None).cast(DecimalType(18, 2))
        ).withColumn(
            "employment_type_normalized",
            F.lit(None).cast(StringType())
        ).select(
            "job_sk",
            "enterprise_job_id",
            "canonical_role_id",
            "company_sk",
            "location_sk",
            "sector_sk",
            "title_normalized",
            "salary_min",
            "salary_max",
            "remote_type",
            "employment_type_normalized",
            "effective_from",
            "effective_to",
            "is_current",
            "record_hash"
        )
        
        # Insert new records
        records_to_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TARGET_TABLE)
        
        print(f"✓ Inserted {inserts + updates} new records")
    else:
        print("✓ No records to insert")
    
    # Write metadata
    metadata_record = spark.createDataFrame([(
        run_id,
        jobs_count,
        inserts,
        updates,
        updates,  # old_records_closed = updates
        no_change,
        force_full_refresh,
        datetime.now(),
        "success",
        None
    )], schema=metadata_schema)
    
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    # Return summary
    summary = {
        "status": "success",
        "run_id": run_id,
        "jobs_extracted": jobs_count,
        "new_jobs_inserted": inserts,
        "jobs_updated": updates,
        "old_records_closed": updates,
        "no_change_jobs": no_change,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(f"\n{json.dumps(summary, indent=2)}")
    
except Exception as e:
    error_msg = str(e)
    print(f"\n✗ Error: {error_msg}")
    
    # Write error metadata
    metadata_record = spark.createDataFrame([(
        run_id,
        jobs_count,
        0,
        0,
        0,
        0,
        force_full_refresh,
        datetime.now(),
        "failed",
        error_msg
    )], schema=metadata_schema)
    
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate job dimension
SELECT 
  COUNT(*) as total_job_versions,
  COUNT(DISTINCT enterprise_job_id) as unique_jobs,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions,
  COUNT(*) - SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as historical_versions,
  AVG(CASE WHEN effective_to IS NULL THEN 0 ELSE DATEDIFF(effective_to, effective_from) END) as avg_version_duration_days,
  SUM(CASE WHEN company_sk = -1 THEN 1 ELSE 0 END) as missing_company_fk,
  SUM(CASE WHEN location_sk = -1 THEN 1 ELSE 0 END) as missing_location_fk
FROM workspace.warehouse.dim_job;

In [0]:
%sql
-- Sample job history (jobs with multiple versions)
SELECT 
  job_sk,
  enterprise_job_id,
  title_normalized,
  company_sk,
  location_sk,
  remote_type,
  effective_from,
  effective_to,
  is_current
FROM workspace.warehouse.dim_job
WHERE enterprise_job_id IN (
  SELECT enterprise_job_id 
  FROM workspace.warehouse.dim_job 
  GROUP BY enterprise_job_id 
  HAVING COUNT(*) > 1 
  LIMIT 10
)
ORDER BY enterprise_job_id, effective_from
LIMIT 50;